### Libraries

In [1]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm

/home/matteo/Documents/Projects/DemocraSci-AM/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")

Using device: cpu


# Speech and Topic

We have a topic assigned to each paragraph from the clustering step.

Given that each paragraph can have multiple sentences, we can split the paragraph into sentences and perform NLI on each sentence against the topic. This way, we can capture more fine-grained stances within a paragraph.

Why not performing clustering on sentences instead of paragraphs? 
- Because we want to maintain the context of the paragraph, which can be important for understanding the stance. Clustering on sentences might lead to losing this context and could result in less coherent topics.

In [3]:
df = pd.read_json("speeches_with_topics.jsonl", lines=True)
df

,Speech_ID,Paragraph_ID,TEXT_SPAN,role,language_of_text,name,Topic,Name,Representation,Representative_Docs,Top_n_words,Representative_document
0,81895,81895_p1,Aber es geht hier noch um andere Fragen. Es ge...,Member of the federal council,DE,Schmid Samuel,108,"108_National Security, Armed Forces, Defense","[National Security, Armed Forces, Defense]",[Der Bundesrat hat zudem am 21. September 2012...,"National Security, Armed Forces, Defense",False
1,81895,81895_p2,"Verschiedene andere Fragen, beispielsweise jen...",Member of the federal council,DE,Schmid Samuel,108,"108_National Security, Armed Forces, Defense","[National Security, Armed Forces, Defense]",[Der Bundesrat hat zudem am 21. September 2012...,"National Security, Armed Forces, Defense",False
2,81895,81895_p3,"Also sind wir bereit, die Analysen zu machen, ...",Member of the federal council,DE,Schmid Samuel,108,"108_National Security, Armed Forces, Defense","[National Security, Armed Forces, Defense]",[Der Bundesrat hat zudem am 21. September 2012...,"National Security, Armed Forces, Defense",False
3,83146,83146_p2,Zum Antrag der Minderheit Maury Pasquier: Dies...,For the committee,DE,Schwaller Urs,67,67_Health insurance costs,[Health insurance costs],[Zu den Anträgen im ersten Block: Bei Artikel ...,Health insurance costs,False
4,83146,83146_p3,Noch eine abschliessende Bemerkung: Wir streic...,For the committee,DE,Schwaller Urs,67,67_Health insurance costs,[Health insurance costs],[Zu den Anträgen im ersten Block: Bei Artikel ...,Health insurance costs,False
...,...,...,...,...,...,...,...,...,...,...,...,...
210223,288681,288681_p8,"En sus, la commission a adopté, par 13 voix co...",For the committee,FR,Python Valentine,149,149_Agriculture and Pesticides,[Agriculture and Pesticides],[Vous avez vu que le Conseil fédéral propose d...,Agriculture and Pesticides,False
210224,288681,288681_p9,Considérant le risque important de contaminati...,For the committee,FR,Python Valentine,68,68_Tobacco and alcohol regulation,[Tobacco and alcohol regulation],[La présente motion du Conseil des Etats conce...,Tobacco and alcohol regulation,False
210225,288681,288681_p11,"En commission, une minorité avait, quant à ell...",For the committee,FR,Python Valentine,68,68_Tobacco and alcohol regulation,[Tobacco and alcohol regulation],[La présente motion du Conseil des Etats conce...,Tobacco and alcohol regulation,False
210226,288681,288681_p12,La proposition individuelle Bäumle déposée hie...,For the committee,FR,Python Valentine,149,149_Agriculture and Pesticides,[Agriculture and Pesticides],[Vous avez vu que le Conseil fédéral propose d...,Agriculture and Pesticides,False


In [4]:
df[["Topic", "Name", "Representation"]].explode(
    "Representation"
).drop_duplicates().reset_index(drop=True).to_csv("unique_topics.csv", index=False)

In [5]:
df_revised = pd.read_csv("unique_topics_withDirections.csv")

In [6]:
df_revised

,Topic,Name,Representation,Representation_with_Direction,Alternatives (stronger in tone)
0,108,"108_National Security, Armed Forces, Defense","National Security, Armed Forces, Defense","Expanding national security, armed forces and ...",NaN
1,67,67_Health insurance costs,Health insurance costs,Reducing health insurance costs,NaN
2,44,44_Federal-Kantonal Finance,Federal-Kantonal Finance,Reducing federal-cantonal finance,NaN
3,50,50_Federalism and cantonal relations,Federalism and cantonal relations,Strengthening cantonal autonomy at the expense...,NaN
4,114,114_Health insurance costs,Health insurance costs,Reducing health insurance costs,NaN
...,...,...,...,...,...
119,12,12_Civil Code and Marriage,Civil Code and Marriage,Modernising civil code to reflect contemporary...,NaN
120,1,1_Technology and Innovation,Technology and Innovation,Increasing public investment in technology and...,NaN
121,6,6_Agriculture and Food Markets,Agriculture and Food Markets,Protecting Swiss agriculture from internationa...,NaN
122,3,3_Government Operations,Government Operations,Streamlining government operations to reduce a...,NaN


In [7]:
merged_df = pd.merge(
    df.drop(columns=["Name", "Representation"]),
    df_revised,
    on="Topic",
    how="left",
)

In [8]:
merged_df

,Speech_ID,Paragraph_ID,TEXT_SPAN,role,language_of_text,name,Topic,Representative_Docs,Top_n_words,Representative_document,Name,Representation,Representation_with_Direction,Alternatives (stronger in tone)
0,81895,81895_p1,Aber es geht hier noch um andere Fragen. Es ge...,Member of the federal council,DE,Schmid Samuel,108,[Der Bundesrat hat zudem am 21. September 2012...,"National Security, Armed Forces, Defense",False,"108_National Security, Armed Forces, Defense","National Security, Armed Forces, Defense","Expanding national security, armed forces and ...",NaN
1,81895,81895_p2,"Verschiedene andere Fragen, beispielsweise jen...",Member of the federal council,DE,Schmid Samuel,108,[Der Bundesrat hat zudem am 21. September 2012...,"National Security, Armed Forces, Defense",False,"108_National Security, Armed Forces, Defense","National Security, Armed Forces, Defense","Expanding national security, armed forces and ...",NaN
2,81895,81895_p3,"Also sind wir bereit, die Analysen zu machen, ...",Member of the federal council,DE,Schmid Samuel,108,[Der Bundesrat hat zudem am 21. September 2012...,"National Security, Armed Forces, Defense",False,"108_National Security, Armed Forces, Defense","National Security, Armed Forces, Defense","Expanding national security, armed forces and ...",NaN
3,83146,83146_p2,Zum Antrag der Minderheit Maury Pasquier: Dies...,For the committee,DE,Schwaller Urs,67,[Zu den Anträgen im ersten Block: Bei Artikel ...,Health insurance costs,False,67_Health insurance costs,Health insurance costs,Reducing health insurance costs,NaN
4,83146,83146_p3,Noch eine abschliessende Bemerkung: Wir streic...,For the committee,DE,Schwaller Urs,67,[Zu den Anträgen im ersten Block: Bei Artikel ...,Health insurance costs,False,67_Health insurance costs,Health insurance costs,Reducing health insurance costs,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
210223,288681,288681_p8,"En sus, la commission a adopté, par 13 voix co...",For the committee,FR,Python Valentine,149,[Vous avez vu que le Conseil fédéral propose d...,Agriculture and Pesticides,False,149_Agriculture and Pesticides,Agriculture and Pesticides,Restricting pesticide use,NaN
210224,288681,288681_p9,Considérant le risque important de contaminati...,For the committee,FR,Python Valentine,68,[La présente motion du Conseil des Etats conce...,Tobacco and alcohol regulation,False,68_Tobacco and alcohol regulation,Tobacco and alcohol regulation,Tightening restrictions on tobacco and alcohol...,NaN
210225,288681,288681_p11,"En commission, une minorité avait, quant à ell...",For the committee,FR,Python Valentine,68,[La présente motion du Conseil des Etats conce...,Tobacco and alcohol regulation,False,68_Tobacco and alcohol regulation,Tobacco and alcohol regulation,Tightening restrictions on tobacco and alcohol...,NaN
210226,288681,288681_p12,La proposition individuelle Bäumle déposée hie...,For the committee,FR,Python Valentine,149,[Vous avez vu que le Conseil fédéral propose d...,Agriculture and Pesticides,False,149_Agriculture and Pesticides,Agriculture and Pesticides,Restricting pesticide use,NaN


# Hypothesis Template

In [9]:
hp_df = merged_df.copy()

In [10]:
TEMPLATE = "The speaker advocates in favor of {}."

In [11]:
hp_df["hypothesis"] = hp_df["Representation_with_Direction"].apply(
    lambda x: TEMPLATE.format(x)
)
hp_df["hypothesis"].values

<ArrowStringArray>
['The speaker advocates in favor of Expanding national security, armed forces and increasing defense expenditures.',
 'The speaker advocates in favor of Expanding national security, armed forces and increasing defense expenditures.',
 'The speaker advocates in favor of Expanding national security, armed forces and increasing defense expenditures.',
                                               'The speaker advocates in favor of Reducing health insurance costs.',
                                               'The speaker advocates in favor of Reducing health insurance costs.',
                                             'The speaker advocates in favor of Reducing federal-cantonal finance.',
        'The speaker advocates in favor of Strengthening cantonal autonomy at the expense of federal coordination.',
                                               'The speaker advocates in favor of Reducing health insurance costs.',
                                             

# NLI Model

In [12]:
model_name = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3).to(
    device
)

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 10917.04it/s]


In [24]:
import numpy as np

batch_size = 1
n_rows = len(hp_df)
model.eval()

results = np.zeros((n_rows, 3))

In [25]:
for i in tqdm(range(0, n_rows, batch_size), desc="NLI Stance Detection"):
    batch_idx = range(i, min(i + batch_size, n_rows))
    batch_premises = hp_df["TEXT_SPAN"].iloc[batch_idx].tolist()
    batch_hypotheses = hp_df["hypothesis"].iloc[batch_idx].tolist()

    inputs = tokenizer(
        batch_premises,
        batch_hypotheses,
        truncation=False,
        padding=False,
        return_tensors="pt",
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)

    results[batch_idx] = probs.cpu().numpy()

NLI Stance Detection:   0%|          | 48/210228 [02:40<195:35:30,  3.35s/it]


KeyboardInterrupt: 

In [26]:
hp_df["Score_Support"] = results[:, 0]
hp_df["Score_Neutral"] = results[:, 1]
hp_df["Score_Oppose"] = results[:, 2]
hp_df["Stance_Intensity"] = hp_df["Score_Support"] - hp_df["Score_Oppose"]

In [32]:
hp_df[hp_df["Stance_Intensity"] > 0.75].head(2)

,Speech_ID,Paragraph_ID,TEXT_SPAN,role,language_of_text,name,Topic,Representative_Docs,Top_n_words,Representative_document,Name,Representation,Representation_with_Direction,Alternatives (stronger in tone),hypothesis,Score_Support,Score_Neutral,Score_Oppose,Stance_Intensity
3,83146,83146_p2,Zum Antrag der Minderheit Maury Pasquier: Dies...,For the committee,DE,Schwaller Urs,67,[Zu den Anträgen im ersten Block: Bei Artikel ...,Health insurance costs,False,67_Health insurance costs,Health insurance costs,Reducing health insurance costs,NaN,The speaker advocates in favor of Reducing hea...,0.970215,0.028229,0.001583,0.968632
4,83146,83146_p3,Noch eine abschliessende Bemerkung: Wir streic...,For the committee,DE,Schwaller Urs,67,[Zu den Anträgen im ersten Block: Bei Artikel ...,Health insurance costs,False,67_Health insurance costs,Health insurance costs,Reducing health insurance costs,NaN,The speaker advocates in favor of Reducing hea...,0.961914,0.036102,0.002020,0.959894


In [33]:
hp_df[hp_df["Stance_Intensity"] < -0.75].head(2)

,Speech_ID,Paragraph_ID,TEXT_SPAN,role,language_of_text,name,Topic,Representative_Docs,Top_n_words,Representative_document,Name,Representation,Representation_with_Direction,Alternatives (stronger in tone),hypothesis,Score_Support,Score_Neutral,Score_Oppose,Stance_Intensity
15,81900,81900_p3,Das schliesslich obsiegende Nein war durch fol...,For the committee,DE,Widmer Hans,108,[Der Bundesrat hat zudem am 21. September 2012...,"National Security, Armed Forces, Defense",False,"108_National Security, Armed Forces, Defense","National Security, Armed Forces, Defense","Expanding national security, armed forces and ...",NaN,The speaker advocates in favor of Expanding na...,0.009933,0.078491,0.911621,-0.901688
16,81904,81904_p1,Politisch zu diskutieren gibt es hier wenig. D...,Member of the federal council,DE,Schmid Samuel,71,[Es wurde gesagt: Im Frühjahr 2009 entschied s...,International Taxation,False,71_International Taxation,International Taxation,Increasing international taxation,NaN,The speaker advocates in favor of Increasing i...,0.027100,0.184814,0.788086,-0.760986


In [30]:
import textwrap


def print_example(idx, w=100):
    print(f"Hypothesis: {textwrap.fill(hp_df.iloc[idx]['hypothesis'], width=w)}")
    print()
    print(f"Text: {textwrap.fill(hp_df.iloc[idx]['TEXT_SPAN'], width=w)}")
    print()
    print("Scores:")
    print(f"\tSupport: {hp_df.iloc[idx]['Score_Support']:.4f}")
    print(f"\tNeutral: {hp_df.iloc[idx]['Score_Neutral']:.4f}")
    print(f"\tOppose: {hp_df.iloc[idx]['Score_Oppose']:.4f}")

In [34]:
print_example(3)

Hypothesis: The speaker advocates in favor of Reducing health insurance costs.

Text: Zum Antrag der Minderheit Maury Pasquier: Diesen Antrag haben wir diskutiert, wir haben auf unsere
Nachfrage auch folgende Präzisierung erhalten: "Im Gesetz spricht man tatsächlich von Zeittarif oder
Pauschaltarif. Zeittarife sind vor allem bei der Spitex üblich, Pauschaltarife beziehen sich eher
auf Tagespauschalen im Pflegeheim." Bei der ursprünglichen Diskussion im Nationalrat hat man die
volle Kostenübernahme nicht nur auf die Pflege im Anschluss an einen Spitalaufenthalt bezogen,
sondern allenfalls auch auf den Fall einer schwerwiegenden Erkrankung; jetzt bezieht es sich klar
nur noch auf den Spitalaufenthalt. Wir haben uns nachher mit 6 zu 2 Stimmen gegen diesen Antrag
ausgesprochen.

Scores:
	Support: 0.9702
	Neutral: 0.0282
	Oppose: 0.0016


In [35]:
print_example(4)

Hypothesis: The speaker advocates in favor of Reducing health insurance costs.

Text: Noch eine abschliessende Bemerkung: Wir streichen ja die Übergangspflege nicht; das wurde
verschiedentlich gesagt. Es ist nicht so, dass wir überhaupt nichts mehr bezahlen wollen, sondern
wir haben für diese Übergangspflege - soweit diese bei einer Rehospitalisierung nicht wieder voll
[PAGE 21] übernommen werden muss - nur einen anderen Schlüssel: 60 Prozent über die Versicherung, 20
Prozent über die öffentliche Hand und maximal 20 Prozent bzw. 20 Franken durch den Privaten. Im
ersten Vorschlag des Nationalrates war vorgesehen, die Kosten der Übergangspflege voll und ganz von
der Versicherung übernehmen zu lassen.

Scores:
	Support: 0.9619
	Neutral: 0.0361
	Oppose: 0.0020


In [36]:
print_example(15)

Hypothesis: The speaker advocates in favor of Expanding national security, armed forces and increasing defense
expenditures.

Text: Das schliesslich obsiegende Nein war durch folgende Argumente motiviert: Durch die Verdoppelung der
Durchdiener könne die Milizarmee gefährdet werden, es müssten zu viele WK-Bataillone aufgehoben
werden; die Reserve werde auf Kosten der aktiven Teile der Armee unverhältnismässig vergrössert; die
Durchführung des Entwicklungsschrittes 2008-2011 werde gefährdet, weil zu viel Unruhe in diesen
Reformprozess hineinkomme, man könne sogar von einer zweiten Reform sprechen. Schliesslich hiess es,
es komme darauf an, Ruhe einkehren zu lassen und zunächst einmal das Riesenprojekt "Armee XXI" zu
konsolidieren. Das waren die Argumente pro und contra.

Scores:
	Support: 0.0099
	Neutral: 0.0785
	Oppose: 0.9116


In [37]:
print_example(16)

Hypothesis: The speaker advocates in favor of Increasing international taxation.

Text: Politisch zu diskutieren gibt es hier wenig. Das Abkommen entspricht einem Standard, der ein Minimum
an Kooperation unter Wahrung der Hoheit des jeweiligen Landes garantiert. Es passt, von dem her
gesehen, durchaus in die Abkommen, die wir bereits mit verschiedenen Ländern abgeschlossen haben.

Scores:
	Support: 0.0271
	Neutral: 0.1848
	Oppose: 0.7881
